# Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
- 1. A schema, including the name of the tool, a description, and\or argument definitions often a JSON schema.
- 2. A function or coroutine to execute

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots can talk. Let me start by recalling what I know about parrots. They\'re birds, right? Some species like parakeets, macaws, and cockatiels are known for their ability to mimic human speech. But why do they do that? Is it just imitation, or is there more to it?\n\nFirst, I should consider their anatomy. I remember that birds have a syrinx, which is different from the human larynx. The syrinx allows them to produce a wide range of sounds. Parrots, especially, have a specialized syrinx that might give them the ability to mimic sounds more accurately. So maybe their vocal anatomy is part of the reason they can talk.\n\nThen there\'s the aspect of social behavior. Parrots are social animals, often living in flocks. In the wild, they might use different calls to communicate with each other, warn of danger, or establish territory. If they\'re in a flock, having a wide range of vocalizations could help them interact more effe

In [2]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # In a real implementation, you would call a weather API here.
    return f"The current weather in {location} is sunny with a high of 25°C."

model_with_tool = model.bind_tools([get_weather])

In [12]:
response =model_with_tool.invoke("What is the weather like in Colima?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

print(response.content)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Colima. Let me check if I have the right tool for that. The available function is get_weather, which requires a location parameter. Colima is a location, so I should use that function. I need to make sure to pass "Colima" as the location argument. Let me structure the tool call correctly.\n', 'tool_calls': [{'id': '4x8g2967c', 'function': {'arguments': '{"location":"Colima"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 157, 'total_tokens': 257, 'completion_time': 0.145229379, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.006131764, 'prompt_tokens_details': None, 'queue_time': 0.119755996, 'total_time': 0.151361143}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'gro

## Tool Execution Loops

In [10]:
# Step 1: Model generate tool calls
messages = [{"role": "user", "content": "What is the weather like in Colima?"}]
ai_msg = model_with_tool.invoke(messages)
print("************* AI message details *************")
print(ai_msg)
messages.append(ai_msg)
print("\n\n************* Messages details *************")
print(messages)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    print("\n\n************* Tool call details *************")
    print(tool_call)
    tool_result = get_weather.invoke(tool_call)
    print("\n\n************* Tool result details *************")
    print(tool_result)
    messages.append(tool_result)
    print("\n\n************* Messages after tool execution details *************")
    print(messages)

# Step 3: Pass results back to model for final response
final_response = model_with_tool.invoke(messages)
print("\n\n************* Final response details *************")
print(final_response)
print("\n\n************* Final response content *************")
print(final_response.text)

************* AI message details *************
content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Colima. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since Colima is a location, I should call that function with "Colima" as the argument. I need to make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'j0vsczr32', 'function': {'arguments': '{"location":"Colima"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 157, 'total_tokens': 257, 'completion_time': 0.135772358, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.006105744, 'prompt_tokens_details': None, 'queue_time': 0.119832474, 'total_time': 0.141878102}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_d58dbe76cd', 'service_tier': 'on_demand', '